# 04 - Fusion, soft combine and full evaluation (v2: all folds)

Runs the whole pipeline on the **out-of-fold predictions** (validation), measures every stage, and tunes the fusion parameters. Nothing here touches the test set.

| Stage | What it adds | Measured by |
|---|---|---|
| A. Each detector alone | baseline | mAP over 14 findings |
| B. WBF of YOLO + Faster R-CNN | fusion of two architectures (grid over IoU and model weights) | mAP over 14 findings |
| B2. Per-class model weights | each class picks its own YOLO : FRCNN weights | mAP over 14 findings + fold check |
| C. + binary head (alpha, t_low) | No finding prediction + down-weighting boxes on normal-looking images | full 15-class mAP |
| D. + multilabel head (beta per class) | per-class presence prior | full 15-class mAP + fold check |
| E. Partner-class copies (check) | only kept if it improves mAP | full 15-class mAP |
| F. TOP_K check | 100 vs 200 predictions per image per model | full 15-class mAP |

**Fold check (B2, D).** A setting chosen separately for every class can fit the validation noise. For each fold k the choices are made on the other four folds and measured on fold k, and compared with one shared setting for all classes chosen the same way. The per-class version is used only if it wins on the held-out folds.

**Inputs:** outputs of 00-03 for **all folds** (the original notebooks + the fold 1-2 and fold 3-4 copies) + `vbd_utils` (v2). CPU is enough.
**Output:** `pipeline_params.json` (used by 05), `oof_final.csv`, `ablation.csv`, `fold_checks.csv`.

In [ ]:
# Import the shared utilities (upload vbd_utils.py as a Kaggle Dataset and attach it)
import os, sys, glob, warnings
def _locate(name, root='/kaggle/input'):
    for cur, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('png1024', 'images', 'labels', 'train', 'test')]
        if name in files:
            return cur
    raise FileNotFoundError(f'{name} not found - attach the vbd-utils dataset')
sys.path.insert(0, _locate('vbd_utils.py'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import vbd_utils as U
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 200)
OUT = '/kaggle/working'
import time

In [ ]:
!pip install -q ensemble-boxes
TOP_K = 100          # predictions kept per image per model; 200 is tested in stage F
TOP_K_ALT = 200
PNG_DIR = U.find_dir('png1024')
folds = pd.read_csv(U.find_file('folds.csv'), index_col=0).fold
targets = pd.read_csv(U.find_file('targets_fused.csv'))
assert hasattr(U, 'select_per_class'), 'old vbd_utils attached - upload the v2 vbd_utils.py to the vbd-utils dataset'

yolo_all = U.top_k_per_image(U.load_concat('oof_yolo_fold*.csv'), TOP_K_ALT)
frcnn_all = U.top_k_per_image(U.load_concat('oof_frcnn_fold*.csv'), TOP_K_ALT)
cls = U.load_concat('oof_cls_fold*.csv').drop_duplicates('image_id').set_index('image_id')

# Evaluate only on folds that all three models have finished
done = set(folds.loc[yolo_all.image_id.unique()]) & set(folds.loc[frcnn_all.image_id.unique()]) & set(folds.loc[cls.index])
ids = folds.index[folds.isin(done)]
yolo_all, frcnn_all = yolo_all[yolo_all.image_id.isin(ids)], frcnn_all[frcnn_all.image_id.isin(ids)]
yolo, frcnn = U.top_k_per_image(yolo_all, TOP_K), U.top_k_per_image(frcnn_all, TOP_K)
cls = cls.loc[ids]
fold_of = folds.loc[ids]
gt = U.build_gt(targets, ids)
print(f'folds used: {sorted(done)} | images: {len(ids)} | abnormal: {gt[gt.class_id != 14].image_id.nunique()}')
fold_checks = []

## A. Each detector alone

In [ ]:
res_yolo = U.evaluate(gt, yolo)
res_frcnn = U.evaluate(gt, frcnn)
ablation = [dict(stage='A1. YOLO', map14=res_yolo['map14'], map15=np.nan),
            dict(stage='A2. Faster R-CNN', map14=res_frcnn['map14'], map15=np.nan)]
ap_stages = {'YOLO': res_yolo['ap'], 'Faster R-CNN': res_frcnn['ap']}

## B. WBF: tune IoU threshold and model weights
Grid over `iou_thr` and the relative weight of the two detectors. On fold 0 the best setting (IoU 0.6, YOLO 2 : FRCNN 1) was at the edge of the old grid, so the grid now goes one step further in both directions. IoU 0.4 was the worst for every weight and is dropped. Every fused set is kept for stage B2.

In [ ]:
from itertools import product
IOUS = [0.5, 0.6, 0.7]
WEIGHTS = [(1, 1), (2, 1), (3, 1), (1, 2)]
fused_cache, grid = {}, []
for iou_thr, w in product(IOUS, WEIGHTS):
    t0 = time.time()
    f = U.wbf_models([yolo, frcnn], weights=list(w), iou_thr=iou_thr, image_ids=ids)
    fused_cache[(iou_thr, w)] = f
    grid.append(dict(iou_thr=iou_thr, w_yolo=w[0], w_frcnn=w[1], map14=U.evaluate(gt, f, verbose=False)['map14']))
    print(grid[-1], f'| {time.time() - t0:.0f}s')
grid = pd.DataFrame(grid).sort_values('map14', ascending=False)
display(grid.round(4))
best_wbf = grid.iloc[0].to_dict()
WBF_IOU = float(best_wbf['iou_thr'])
fused = fused_cache[(WBF_IOU, (int(best_wbf['w_yolo']), int(best_wbf['w_frcnn'])))]
res_fused = U.evaluate(gt, fused)
ablation.append(dict(stage='B. WBF (YOLO + FRCNN)', map14=res_fused['map14'], map15=np.nan))
ap_stages['WBF'] = res_fused['ap']

## B2. Per-class model weights
At the chosen IoU, every class picks the YOLO : FRCNN weights with the best AP (on fold 0, Pneumothorax lost 0.05 AP in the fusion because YOLO is weak on it). WBF never merges boxes of different classes, so this is the same as one fusion per class.

In [ ]:
cands_w = {f'{a}:{b}': fused_cache[(WBF_IOU, (a, b))] for a, b in WEIGHTS}
sel_w = U.select_per_class(cands_w, gt, fold_of)
display(sel_w['ap'].round(4).assign(chosen=sel_w['choice']))
print('Fold check - choices made on the other folds, measured on this fold (mAP over 14 findings):')
display(sel_w['check'].round(4))
USE_PC_WEIGHTS = len(sel_w['check']) > 0 and sel_w['check'].gain.mean() > 0
print(f"mean gain of per-class weights on held-out folds: {sel_w['check'].gain.mean() if len(sel_w['check']) else float('nan'):+.4f}"
      f' -> use per-class weights: {USE_PC_WEIGHTS}')
if USE_PC_WEIGHTS:
    W_PC = [[float(x) for x in sel_w['choice'][n].split(':')] for n in U.CLASSES]
    fused = U.assemble_per_class(cands_w, sel_w['choice'])
    res_fused = U.evaluate(gt, fused)
    ablation.append(dict(stage='B2. per-class model weights', map14=res_fused['map14'], map15=np.nan))
    ap_stages['WBF per-class weights'] = res_fused['ap']
else:
    W_PC = [[float(best_wbf['w_yolo']), float(best_wbf['w_frcnn'])]] * 14
fold_checks.append(sel_w['check'].assign(stage='B2 model weights'))

## C. Soft combine with the binary head
Every image gets a No finding prediction with score `1 - p_abn`. Box scores are multiplied by `p_abn ** alpha`; images with `p_abn < t_low` keep only No finding.

In [ ]:
rows = []
for a in [0, 0.25, 0.5, 0.75, 1, 2]:
    r = U.evaluate(gt, U.soft_combine(fused, cls, alpha=a), verbose=False)
    rows.append(dict(alpha=a, map15=r['map15'], map14=r['map14'], AP_no_finding=r['ap']['No finding']))
alpha_tab = pd.DataFrame(rows); display(alpha_tab.round(4))
ALPHA = float(alpha_tab.sort_values('map15', ascending=False).alpha.iloc[0])

rows = []
for t in [0, 0.01, 0.02, 0.05, 0.1, 0.2]:
    r = U.evaluate(gt, U.soft_combine(fused, cls, alpha=ALPHA, t_low=t), verbose=False)
    rows.append(dict(t_low=t, map15=r['map15'], map14=r['map14'],
                     images_emptied=int((cls.p_abn < t).sum()),
                     abnormal_images_emptied=int(((cls.p_abn < t) & cls.index.isin(targets.image_id)).sum())))
tlow_tab = pd.DataFrame(rows); display(tlow_tab.round(4))
# a hard gate empties images, so it must earn its place: > 0 only if it adds at least 0.001
best_t = tlow_tab.sort_values('map15', ascending=False).iloc[0]
T_LOW = float(best_t.t_low) if best_t.map15 >= tlow_tab.map15.iloc[0] + 0.001 else 0.0
print(f'chosen alpha={ALPHA}, t_low={T_LOW}')

res_c = U.evaluate(gt, U.soft_combine(fused, cls, alpha=ALPHA, t_low=T_LOW))
ablation.append(dict(stage='C. + binary head', map14=res_c['map14'], map15=res_c['map15']))
ap_stages['+ binary head'] = res_c['ap']

The column `abnormal_images_emptied` is the price of `t_low`: abnormal images that lose **all** their boxes. It is the reason we combine softly instead of gating.

## D. Soft combine with the multilabel head
Box scores of class c are multiplied by `p_c ** beta[c]`. On fold 0, beta = 2 was never chosen, so the grid is now finer between 0 and 1. The same fold check as in B2 decides between one beta for all classes and one per class.

In [ ]:
BETAS = [0, 0.25, 0.5, 0.75, 1]
cands_b = {b: U.soft_combine(fused, cls, alpha=ALPHA, beta=np.full(14, b), t_low=T_LOW) for b in BETAS}
sel_b = U.select_per_class(cands_b, gt, fold_of)
display(sel_b['ap'].round(4).assign(chosen=sel_b['choice']))
print('Fold check (mAP over 14 findings):')
display(sel_b['check'].round(4))
USE_PC_BETA = len(sel_b['check']) > 0 and sel_b['check'].gain.mean() > 0
print(f"mean gain of per-class beta on held-out folds: {sel_b['check'].gain.mean() if len(sel_b['check']) else float('nan'):+.4f}"
      f' -> use per-class beta: {USE_PC_BETA}')
if USE_PC_BETA:
    BETA = np.array([float(sel_b['choice'][n]) for n in U.CLASSES])
else:
    BETA = np.full(14, float(sel_b['ap'].mean().idxmax()))
print('beta:', dict(zip(U.CLASSES, BETA)))
fold_checks.append(sel_b['check'].assign(stage='D beta'))

final = U.soft_combine(fused, cls, alpha=ALPHA, beta=BETA, t_low=T_LOW)
res_d = U.evaluate(gt, final)
ablation.append(dict(stage='D. + multilabel head', map14=res_d['map14'], map15=res_d['map15']))
ap_stages['+ multilabel head'] = res_d['ap']

## E. Partner-class copies (decision check)
First: for GT regions labeled with both classes of a pair, how often does the pipeline already predict both?
Then: does adding a partner copy at a reduced score improve mAP? The copies are kept **only** if they do.

In [ ]:
display(U.double_label_coverage(gt, final, score_thr=0.1))
rows, USE_PARTNER, PARTNER_FACTOR = [], False, None
pair_classes = [U.ALL_CLASSES[c] for p in U.PARTNER_PAIRS for c in p]
for f in [0.3, 0.5, 0.7]:
    r = U.evaluate(gt, U.add_partner_copies(final, factor=f), verbose=False)
    rows.append(dict(factor=f, map15=r['map15'], **{n: r['ap'][n] for n in pair_classes}))
part = pd.DataFrame(rows)
part.loc[len(part)] = dict(factor='none', map15=res_d['map15'], **{n: res_d['ap'][n] for n in pair_classes})
display(part.round(4))
best_p = part[part.factor != 'none'].sort_values('map15', ascending=False).iloc[0]
res_e = res_d
if best_p.map15 > res_d['map15'] + 1e-4:
    USE_PARTNER, PARTNER_FACTOR = True, float(best_p.factor)
    final = U.add_partner_copies(final, factor=PARTNER_FACTOR)
    res_e = U.evaluate(gt, final)
    ablation.append(dict(stage='E. + partner copies', map14=res_e['map14'], map15=res_e['map15']))
    ap_stages['+ partner copies'] = res_e['ap']
print('use partner copies:', USE_PARTNER, PARTNER_FACTOR)

## F. TOP_K check
Rebuilds the whole chosen pipeline from the raw predictions with `wbf_per_class` - the exact function notebook 05 uses on the test set - for TOP_K = 100 and 200. The TOP_K = 100 run must reproduce the mAP above (a check that 05 will do the same as 04). 200 is kept only if it adds at least 0.001.

In [ ]:
def build_pipeline(top_k):
    y, f = U.top_k_per_image(yolo_all, top_k), U.top_k_per_image(frcnn_all, top_k)
    fu = U.wbf_per_class([y, f], W_PC, iou_thr=WBF_IOU, image_ids=ids)
    out = U.soft_combine(fu, cls, alpha=ALPHA, beta=BETA, t_low=T_LOW)
    return U.add_partner_copies(out, factor=PARTNER_FACTOR) if USE_PARTNER else out

rebuilt = build_pipeline(TOP_K)
r100 = U.evaluate(gt, rebuilt, verbose=False)
print(f"rebuilt with TOP_K={TOP_K}: mAP15 {r100['map15']:.4f} (stage E: {res_e['map15']:.4f})")
assert abs(r100['map15'] - res_e['map15']) < 1e-6, 'rebuild does not match - 05 would differ from 04'
alt = build_pipeline(TOP_K_ALT)
r200 = U.evaluate(gt, alt, verbose=False)
print(f"TOP_K={TOP_K_ALT}: mAP15 {r200['map15']:.4f} | mAP14 {r200['map14']:.4f}")
if r200['map15'] >= r100['map15'] + 0.001:
    TOP_K, final = TOP_K_ALT, alt
    ablation.append(dict(stage=f'F. TOP_K = {TOP_K_ALT}', map14=r200['map14'], map15=r200['map15']))
    ap_stages[f'TOP_K {TOP_K_ALT}'] = r200['ap']
print('TOP_K:', TOP_K)

## Ablation: what each stage contributes

In [ ]:
ablation = pd.DataFrame(ablation)
ablation['delta_map15'] = ablation.map15.diff()
display(ablation.round(4)); ablation.to_csv(f'{OUT}/ablation.csv', index=False)
display(U.plot_ap_table(ap_stages, 'AP per class at each stage'))
fold_checks = pd.concat(fold_checks, ignore_index=True)
fold_checks.to_csv(f'{OUT}/fold_checks.csv', index=False)

## Full report on the final pipeline
PR curves, detection confusion matrix, recall by box size and by radiologist agreement, and example images.

In [ ]:
res_final = U.full_detection_report(gt, final, PNG_DIR, 'Final pipeline (out-of-fold)', score_thr=0.2)

In [ ]:
# Per-fold stability: a large spread means the numbers above should be read with care
per_fold = {k: U.evaluate(U.build_gt(targets, ids[(fold_of == k).values]),
                          final[final.image_id.isin(ids[(fold_of == k).values])], verbose=False)['map15']
            for k in sorted(done)}
print('mAP (15 classes) per fold:', {k: round(v, 4) for k, v in per_fold.items()})
print(f'mean {np.mean(list(per_fold.values())):.4f} | std {np.std(list(per_fold.values())):.4f}')

## Save the tuned parameters for the submission notebook

In [ ]:
params = dict(version=2, top_k=TOP_K, wbf_iou=WBF_IOU, w_yolo=float(best_wbf['w_yolo']), w_frcnn=float(best_wbf['w_frcnn']),
              use_pc_weights=bool(USE_PC_WEIGHTS), w_per_class=W_PC,
              alpha=ALPHA, beta=BETA.tolist(), use_pc_beta=bool(USE_PC_BETA), t_low=T_LOW,
              use_partner=USE_PARTNER, partner_factor=PARTNER_FACTOR,
              oof_map15=res_final['map15'], oof_map14=res_final['map14'], folds_used=sorted(int(k) for k in done))
U.save_json(params, f'{OUT}/pipeline_params.json')
final.to_csv(f'{OUT}/oof_final.csv', index=False)
params